# 📘 Week 3 · Day 17-19: Gradient Boosting 3대장 — XGBoost · LightGBM · CatBoost

## 🎯 학습 목표
- Gradient Boosting의 **수학적 원리** 이해
- XGBoost, LightGBM, CatBoost **각각의 철학과 차이점**
- 실전에서 **어떤 상황에 무엇을 쓸지** 감 잡기
- 주요 **하이퍼파라미터와 튜닝 전략**
- early stopping, 범주형 처리, 피처 중요도

> 🏆 Kaggle tabular 대회 **90% 이상의 우승 솔루션**이 이 셋 중 하나 혹은 앙상블.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.datasets import make_classification, fetch_california_housing
from sklearn.metrics import roc_auc_score, mean_squared_error
import warnings; warnings.filterwarnings('ignore')
np.random.seed(0)

---

## 1. Gradient Boosting이란?

### 직관
**약한 모델(얕은 트리) 여러 개를 순차적으로** 학습시켜 전 단계의 **잔차(오차)를 다음 모델이 보완**하도록 만듭니다.

```
F₀(x) = 초기 예측 (평균)
F₁(x) = F₀(x) + γ₁ · h₁(x)   ← h₁은 F₀ 오차를 학습
F₂(x) = F₁(x) + γ₂ · h₂(x)   ← h₂는 F₁ 오차를 학습
...
Fₘ(x) = 최종
```

### Random Forest vs Gradient Boosting
| 항목 | RF | GBM |
|------|----|----|
| 학습 방식 | **병렬** (독립) | **순차** (의존) |
| 각 트리 | 깊고 다양 | 얕고 weak |
| 목표 | 분산 감소 | 편향 감소 |
| 속도 | 병렬 가능 | 더 느림 |
| 튜닝 | 거의 불필요 | 하이퍼파라미터 중요 |


In [ ]:
# Gradient Boosting을 밑바닥부터 구현
from sklearn.tree import DecisionTreeRegressor

X, y = make_classification(n_samples=500, random_state=0)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=0)

# 간단한 GBM (회귀 손실로 근사)
def simple_gbm(X_tr, y_tr, X_te, n_trees=100, lr=0.1, max_depth=3):
    F_train = np.full(len(y_tr), y_tr.mean())
    F_test  = np.full(len(X_te), y_tr.mean())
    trees = []
    for _ in range(n_trees):
        residual = y_tr - F_train
        t = DecisionTreeRegressor(max_depth=max_depth, random_state=0)
        t.fit(X_tr, residual)
        F_train += lr * t.predict(X_tr)
        F_test  += lr * t.predict(X_te)
        trees.append(t)
    return F_test

pred = simple_gbm(X_tr, y_tr, X_te, n_trees=100, lr=0.1)
pred_class = (pred > 0.5).astype(int)
print(f"직접 구현한 GBM Accuracy: {(pred_class == y_te).mean():.4f}")

---

## 2. XGBoost

### 특징
- 2014년 등장, Kaggle을 지배한 첫 GBM 라이브러리
- **2차 도함수(Hessian)** 사용 → 더 정확한 최적화
- L1/L2 regularization 내장
- GPU 지원

### 주요 하이퍼파라미터

| 파라미터 | 역할 | 추천 범위 |
|----------|------|-----------|
| `n_estimators` | 트리 개수 | 500~5000 (early stopping 사용) |
| `learning_rate` | 한 번에 반영하는 비율 | 0.01~0.1 |
| `max_depth` | 트리 깊이 | 3~8 |
| `min_child_weight` | 리프 최소 헤시안 | 1~10 |
| `subsample` | 행 샘플링 비율 | 0.6~0.9 |
| `colsample_bytree` | 피처 샘플링 | 0.6~0.9 |
| `reg_alpha` | L1 | 0~10 |
| `reg_lambda` | L2 | 0~10 |
| `gamma` | 최소 손실 감소 | 0~5 |


In [ ]:
import xgboost as xgb

X, y = make_classification(n_samples=5000, n_features=20, n_informative=10, random_state=0)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=0, stratify=y)

# 기본 사용
model = xgb.XGBClassifier(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    min_child_weight=1,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1.0,
    eval_metric='auc',
    early_stopping_rounds=50,
    random_state=0,
    n_jobs=-1
)

model.fit(X_tr, y_tr, eval_set=[(X_te, y_te)], verbose=False)

y_pred_prob = model.predict_proba(X_te)[:, 1]
print(f"Best iteration: {model.best_iteration}")
print(f"AUC: {roc_auc_score(y_te, y_pred_prob):.4f}")

In [ ]:
# 피처 중요도 시각화
xgb.plot_importance(model, max_num_features=15, importance_type='gain', height=0.5)
plt.title('XGBoost Feature Importance (Gain)')
plt.tight_layout(); plt.show()

### XGBoost - Early Stopping 시각화

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
results = model.evals_result()
ax.plot(results['validation_0']['auc'])
ax.axvline(model.best_iteration, color='red', ls='--', label=f'best={model.best_iteration}')
ax.set_xlabel('iteration'); ax.set_ylabel('AUC'); ax.set_title('XGBoost validation AUC')
ax.legend(); plt.show()

---

## 3. LightGBM — 가장 빠르고 강력

### 특징
- Microsoft에서 개발 (2017)
- **Leaf-wise** 분할 (XGBoost는 level-wise) → 더 깊고 빠름
- **Histogram 기반** 분할 (큰 데이터에 강함)
- **범주형 변수 직접 처리** (label encoding 후 `categorical_feature` 지정)
- 메모리 효율 최고

### 주요 하이퍼파라미터 (XGBoost와 이름 다름)

| 파라미터 | 역할 |
|----------|------|
| `num_leaves` | 리프 최대 개수 (가장 중요!) — 보통 31~255 |
| `max_depth` | 보통 -1 (제한 없음) |
| `min_child_samples` | 리프 최소 샘플 수 |
| `feature_fraction` | colsample_bytree와 동일 |
| `bagging_fraction` | subsample과 동일 |
| `bagging_freq` | bagging 주기 |


In [ ]:
import lightgbm as lgb

lgb_model = lgb.LGBMClassifier(
    n_estimators=1000,
    learning_rate=0.05,
    num_leaves=63,
    max_depth=-1,
    min_child_samples=20,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=0.1,
    random_state=0,
    n_jobs=-1,
    verbose=-1
)

lgb_model.fit(
    X_tr, y_tr,
    eval_set=[(X_te, y_te)],
    eval_metric='auc',
    callbacks=[lgb.early_stopping(50), lgb.log_evaluation(0)]
)

y_pred_prob = lgb_model.predict_proba(X_te)[:, 1]
print(f"Best iteration: {lgb_model.best_iteration_}")
print(f"AUC: {roc_auc_score(y_te, y_pred_prob):.4f}")

In [ ]:
# 피처 중요도
lgb.plot_importance(lgb_model, max_num_features=15, importance_type='gain', height=0.5)
plt.title('LightGBM Feature Importance (Gain)')
plt.tight_layout(); plt.show()

### 🔑 LightGBM leaf-wise의 함정

`num_leaves`가 너무 크면 과적합 매우 쉬움. 권장: `num_leaves < 2^max_depth`

In [ ]:
# num_leaves에 따른 성능 변화
from sklearn.model_selection import cross_val_score

for nl in [15, 31, 63, 127, 255]:
    m = lgb.LGBMClassifier(n_estimators=200, num_leaves=nl, random_state=0, verbose=-1, n_jobs=-1)
    s = cross_val_score(m, X_tr, y_tr, cv=3, scoring='roc_auc', n_jobs=-1).mean()
    print(f"num_leaves={nl:3d} → AUC={s:.4f}")

---

## 4. CatBoost — 범주형 데이터 특화

### 특징
- Yandex 개발 (2017)
- **Ordered Boosting** — target leakage 방지
- **범주형 변수 자동 처리** (Target Encoding 내장)
- 기본값이 좋아 튜닝 없이도 성능 좋음
- GPU 지원, 인자 적음

In [ ]:
from catboost import CatBoostClassifier

cb_model = CatBoostClassifier(
    iterations=1000,
    learning_rate=0.05,
    depth=6,
    l2_leaf_reg=3,
    random_seed=0,
    verbose=0,
    early_stopping_rounds=50
)

cb_model.fit(X_tr, y_tr, eval_set=(X_te, y_te), verbose=0)
y_pred_prob = cb_model.predict_proba(X_te)[:, 1]
print(f"Best iteration: {cb_model.best_iteration_}")
print(f"AUC: {roc_auc_score(y_te, y_pred_prob):.4f}")

In [ ]:
# 피처 중요도
fi = pd.Series(cb_model.feature_importances_, index=[f'f{i}' for i in range(X_tr.shape[1])])
fi.sort_values().tail(15).plot(kind='barh', figsize=(8, 5), color='teal')
plt.title('CatBoost Feature Importance'); plt.tight_layout(); plt.show()

---

## 5. 3대장 비교 & 선택 가이드

| 항목 | XGBoost | LightGBM | CatBoost |
|------|---------|----------|----------|
| 속도 | 중간 | **최고** | 중간 |
| 메모리 | 중간 | **최고** | 큼 |
| 범주형 처리 | 수동 | label enc. 후 지정 | **자동, 최고** |
| 작은 데이터 | 좋음 | 과적합 위험 | **가장 안정** |
| 큰 데이터 | 느릴 수 있음 | **최적** | 중간 |
| 튜닝 민감도 | 중간 | **높음** (num_leaves) | 낮음 |
| 기본값 성능 | 보통 | 보통 | **최고** |

### 추천 워크플로우
1. **첫 베이스라인**: CatBoost (튜닝 없이도 good)
2. **메인 모델**: LightGBM (튜닝으로 최고점)
3. **앙상블 재료**: XGBoost 추가


---

## 6. 3대장 실전 비교 (같은 데이터에)

In [ ]:
# 캘리포니아 주택 회귀 문제로 비교
from sklearn.datasets import fetch_california_housing
housing = fetch_california_housing()
X, y = housing.data, housing.target
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=0)

results = {}

# XGBoost
xm = xgb.XGBRegressor(n_estimators=500, learning_rate=0.05, max_depth=6,
                     early_stopping_rounds=30, random_state=0, n_jobs=-1)
xm.fit(X_tr, y_tr, eval_set=[(X_te, y_te)], verbose=False)
results['XGBoost']  = np.sqrt(mean_squared_error(y_te, xm.predict(X_te)))

# LightGBM
lm = lgb.LGBMRegressor(n_estimators=500, learning_rate=0.05, num_leaves=63, random_state=0, verbose=-1, n_jobs=-1)
lm.fit(X_tr, y_tr, eval_set=[(X_te, y_te)], callbacks=[lgb.early_stopping(30), lgb.log_evaluation(0)])
results['LightGBM'] = np.sqrt(mean_squared_error(y_te, lm.predict(X_te)))

# CatBoost
from catboost import CatBoostRegressor
cm = CatBoostRegressor(iterations=500, learning_rate=0.05, depth=6, verbose=0, random_seed=0, early_stopping_rounds=30)
cm.fit(X_tr, y_tr, eval_set=(X_te, y_te), verbose=0)
results['CatBoost'] = np.sqrt(mean_squared_error(y_te, cm.predict(X_te)))

pd.Series(results).sort_values().to_frame('RMSE').round(4)

---

## 7. 튜닝 시 주요 전략

### 7.1 단계별 튜닝
1. **`learning_rate=0.1`** 고정, `n_estimators`는 early stopping으로 자동 결정
2. **트리 구조** 튜닝: `max_depth`, `num_leaves`, `min_child_samples`
3. **샘플링** 튜닝: `subsample`, `colsample_bytree`
4. **정규화** 튜닝: `reg_alpha`, `reg_lambda`
5. **`learning_rate`를 0.01~0.02로 낮추고** `n_estimators` 키움 (final fit)

### 7.2 과적합 방지 체크리스트
- [ ] early stopping 사용
- [ ] subsample < 1.0
- [ ] colsample_bytree < 1.0
- [ ] min_child_weight 크게
- [ ] reg_alpha, reg_lambda 증가
- [ ] max_depth / num_leaves 줄이기


---

## 8. 실전 템플릿 — LightGBM K-Fold + 피처 중요도

In [ ]:
def train_lgb_kfold(X, y, params, n_splits=5, random_state=0):
    oof  = np.zeros(len(y))
    feat_imp = np.zeros(X.shape[1])
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)

    for fold, (tr, val) in enumerate(skf.split(X, y)):
        X_tr, X_val = X[tr], X[val]
        y_tr, y_val = y[tr], y[val]

        model = lgb.LGBMClassifier(**params)
        model.fit(X_tr, y_tr,
                  eval_set=[(X_val, y_val)],
                  callbacks=[lgb.early_stopping(50), lgb.log_evaluation(0)])

        oof[val] = model.predict_proba(X_val)[:, 1]
        feat_imp += model.feature_importances_ / n_splits
        print(f"Fold {fold+1}: AUC={roc_auc_score(y_val, oof[val]):.4f}")

    print(f"\nOverall OOF AUC: {roc_auc_score(y, oof):.4f}")
    return oof, feat_imp

X, y = make_classification(n_samples=5000, n_features=20, random_state=0)
params = dict(n_estimators=500, learning_rate=0.05, num_leaves=63,
              subsample=0.8, colsample_bytree=0.8,
              random_state=0, verbose=-1, n_jobs=-1)
oof, fi = train_lgb_kfold(X, y, params)

---

## 📝 Day 17-19 체크리스트
- [ ] Gradient Boosting 원리 이해
- [ ] XGBoost/LightGBM/CatBoost 각각 학습 돌려봄
- [ ] early stopping 사용
- [ ] 하이퍼파라미터별 역할 이해
- [ ] K-Fold + LightGBM 템플릿 체화

다음은 **하이퍼파라미터 튜닝 (Optuna)** 과 **앙상블 / 스태킹**입니다.